# Student Academic Performance Analysis
Synthetic data only. This notebook reuses the same cleaning, scoring and charts as the app. Correlations are descriptive, not causal.

In [ ]:
from pathlib import Path
import sys
project_root = Path.cwd().resolve()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
if not (project_root / "src").is_dir():
    raise RuntimeError("Open this notebook from the repository or notebooks directory.")
sys.path.insert(0, str(project_root))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import asdict
from src.data_cleaning import load_csv, clean_data, SUBJECT_COLUMNS
from src.analysis import calculate_performance, summary_metrics, group_performance, correlation, GRADE_ORDER
from src.visualizations import bar_chart, scatter_chart
raw = load_csv(project_root / "data/student_performance.csv")
cleaned, report = clean_data(raw)
students = calculate_performance(cleaned)
print(asdict(report))
students.head()

In [ ]:
summary_metrics(students)

In [ ]:
students.groupby("grade").academic_score.agg(["count", "mean", "min", "max"]).reindex(GRADE_ORDER)

In [ ]:
means = students[SUBJECT_COLUMNS].mean()
fig = bar_chart([x.replace("_score", "").title() for x in means.index], means.values, "Subject averages", "Mean score")
plt.show()
plt.close(fig)

In [ ]:
for field in ["department", "semester"]:
    print(group_performance(students, field).round(2).to_string(index=False))
for field in ["attendance_percentage", "study_hours_per_week"]:
    print(field, correlation(students, field))
    fig = scatter_chart(students, field, field.replace("_", " ").title(), "Association with academic score")
    plt.show()
    plt.close(fig)

In [ ]:
students.loc[students.at_risk, ["student_id", "academic_score", "attendance_percentage", "risk_reason", "score_imputed"]].head(15)

In [ ]:
students.sort_values("academic_score", ascending=False)[["student_id", "name", "academic_score", "grade", "score_imputed"]].head(10)

## Interpretation limits
The fictional generator deliberately shares variation among attendance, study time and scores. Grades depend on a demonstration rubric. Median imputation can change ranks and flags; inspect the quality report before interpretation. No model or causal claim is made.